In [9]:
import duckdb
import pandas as pd

# Connect to existing database
# con = duckdb.connect(r"C:\Users\shaog\OneDrive\Documents\DSE3101\mind-the-data-gap\data.duckdb")
con = duckdb.connect("data.duckdb")

In [13]:
# Check tables
print(con.execute("SHOW TABLES").fetchdf())

          name
0  abt_pt_jrny
1  abt_pt_ride
2     bus_stop
3      pt_jrny
4      pt_ride
5   vls_marker


## Cleaning pt_ride
1. Drop incomplete data
2. Pick a representative weekday (13/2/2025)
3. Drop irrelevant columns
4. Select relevant patron categories

In [14]:
# Check columns of pt_ride first
print(con.execute("DESCRIBE pt_ride").fetchdf())

           column_name column_type null   key default extra
0               BIZ_DT        DATE  YES  None    None  None
1          BUS_SVC_NUM      BIGINT  YES  None    None  None
2              CRD_NUM     VARCHAR  YES  None    None  None
3      DEST_LOC_ID_NUM      BIGINT  YES  None    None  None
4             ENTRY_DT        DATE  YES  None    None  None
5             ENTRY_TM        TIME  YES  None    None  None
6              EXIT_DT        DATE  YES  None    None  None
7              EXIT_TM        TIME  YES  None    None  None
8          JRNY_ID_NUM      BIGINT  YES  None    None  None
9      ORIG_LOC_ID_NUM      BIGINT  YES  None    None  None
10  PATRON_CATG_ID_NUM      BIGINT  YES  None    None  None
11              PAY_CD      BIGINT  YES  None    None  None
12       RIDE_DISC_AMT      DOUBLE  YES  None    None  None
13    RIDE_DIST_KM_CNT      DOUBLE  YES  None    None  None
14       RIDE_FARE_AMT      DOUBLE  YES  None    None  None
15         RIDE_ID_NUM      BIGINT  YES 

In [ ]:
# Select representative weekday, drop incomplete rows and keep perfect rides
temp_pt_ride = con.execute("""
    SELECT 
        CRD_NUM,
        JRNY_ID_NUM,
        BUS_SVC_NUM,
        ENTRY_DT,
        ENTRY_TM,
        EXIT_DT,
        EXIT_TM,
        ORIG_LOC_ID_NUM,
        DEST_LOC_ID_NUM,
        TRNSPT_MODE_CD,
        PATRON_CATG_ID_NUM,
        RIDE_FARE_AMT,
        RIDE_DIST_KM_CNT,
        RIDE_MIN_CNT,
        RIDE_ST_CD
    FROM pt_ride
    WHERE BIZ_DT = '2025-02-13'
      AND CRD_NUM IS NOT NULL
      AND ORIG_LOC_ID_NUM IS NOT NULL
      AND DEST_LOC_ID_NUM IS NOT NULL
      AND ENTRY_TM IS NOT NULL
      AND EXIT_TM IS NOT NULL
      AND RIDE_FARE_AMT IS NOT NULL
      AND RIDE_MIN_CNT IS NOT NULL
      AND RIDE_ST_CD = 1
""")

In [ ]:
temp_pt_ride_df = temp_pt_ride.fetchdf()
print(temp_pt_ride_df.shape) # (3727069, 15)
# print(temp_pt_ride.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3727069, 15)


In [ ]:
print(temp_pt_ride['PATRON_CATG_ID_NUM'].value_counts().sort_index())

PATRON_CATG_ID_NUM
0        837
1    1665312
2      12397
3     728162
4    1161122
5      15999
6     143240
Name: count, dtype: int64


In [17]:
# Select representative weekday, drop incomplete rows, keep perfect rides, filter for certain patron categories and join the mappings
cleaned_pt_ride = con.execute("""
    SELECT 
        pr.crd_num,
        pr.jrny_id_num,
        pr.bus_svc_num,
        pr.entry_dt,
        pr.entry_tm,
        pr.exit_dt,
        pr.exit_tm,
        pr.orig_loc_id_num,
        pr.dest_loc_id_num,
        pr.trnspt_mode_cd,
        m.trnspt_mode_desc_txt AS trnspt_mode,
        pr.patron_catg_id_num,
        p.patron_catg_desc_txt AS patron_catg,
        pr.ride_fare_amt,
        pr.ride_dist_km_cnt,
        pr.ride_min_cnt,
        pr.ride_st_cd,
        st.st_cd_desc_txt AS ride_st
    FROM pt_ride pr
	JOIN trnspt_mode_cd m using (trnspt_mode_cd)
	JOIN patron_catg_id_num p using (patron_catg_id_num)
	JOIN jrny_st_cd_ride_st_cd st on pr.ride_st_cd = st.st_cd
    WHERE pr.biz_dt = '2025-02-13'
      AND pr.crd_num IS NOT NULL
      AND pr.orig_loc_id_num IS NOT NULL
      AND pr.dest_loc_id_num IS NOT NULL
      AND pr.entry_tm IS NOT NULL
      AND pr.exit_tm IS NOT NULL
      AND pr.ride_fare_amt IS NOT NULL
      AND pr.ride_min_cnt IS NOT NULL
      AND pr.ride_st_cd = 1
      AND pr.patron_catg_id_num IN (1, 3, 4)
""")

#pd.read_sql("SELECT * FROM temp_results", con) # to directly manipulate df but I keep the sql form updated for now first

In [ ]:
cleaned_pt_ride_df = cleaned_pt_ride.fetchdf()
print(cleaned_pt_ride_df.shape) # (3554596, 18)
# print(cleaned_pt_ride.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(3554596, 18)


In [19]:
con.close()